## Step 1: Load data, model, and this season's remaining fixtures

## Step 1b: Fetch the full season's fixture list (preseason mode)
Every fixture, played or not — used later to simulate the season from a blank slate.

In [1]:
import os
import pandas as pd
import requests
import joblib
from dotenv import load_dotenv

load_dotenv() 

FOOTBALL_DATA_API_KEY = os.environ["FOOTBALL_DATA_API_KEY"]
CURRENT_SEASON = "2026"

features = pd.read_csv("../data/features/match_features_multi_season.csv", parse_dates=["utcDate"])

clean_matches = pd.read_csv("../data/processed/clean_matches_multi_season.csv", parse_dates=["utcDate"])
clean_matches = clean_matches.sort_values("utcDate").reset_index(drop=True)

model = joblib.load("../data/features/logistic_regression_model.pkl")

def fetch_scheduled_fixtures():
    url = "https://api.football-data.org/v4/competitions/PL/matches"
    headers = {"X-Auth-Token": FOOTBALL_DATA_API_KEY}
    params = {"status": "SCHEDULED"}

    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()
    matches = response.json()["matches"]

    fixtures = pd.DataFrame([{
        "match_id": m["id"],
        "utcDate": m["utcDate"],
        "matchday": m["matchday"],
        "season": m["season"]["startDate"][:4],
        "home_team": m["homeTeam"]["name"],
        "away_team": m["awayTeam"]["name"],
    } for m in matches])

    fixtures["utcDate"] = pd.to_datetime(fixtures["utcDate"])
    fixtures = fixtures[fixtures["season"] == CURRENT_SEASON]
    fixtures = fixtures.sort_values("utcDate").reset_index(drop=True)
    return fixtures

remaining_fixtures = fetch_scheduled_fixtures()
print(f"Remaining fixtures this season: {len(remaining_fixtures)}")
remaining_fixtures.head()

Remaining fixtures this season: 350


,match_id,utcDate,matchday,season,home_team,away_team
0,560572,2026-09-12 14:00:00+00:00,4,2026,Crystal Palace FC,Ipswich Town FC
1,560573,2026-09-12 14:00:00+00:00,4,2026,Liverpool FC,Fulham FC
2,560574,2026-09-12 14:00:00+00:00,4,2026,Aston Villa FC,Nottingham Forest FC
3,560576,2026-09-12 14:00:00+00:00,4,2026,AFC Bournemouth,Brentford FC
4,560581,2026-09-12 14:00:00+00:00,4,2026,Chelsea FC,Hull City AFC


## Step 2b: Build preseason baseline state
Computes each team's historical final standings, derives a promoted-team fallback position from the dataset itself, and zero-initializes every team for the preseason simulation (no real in-season results used).

In [2]:
def fetch_full_season_fixtures():
    url = "https://api.football-data.org/v4/competitions/PL/matches"
    headers = {"X-Auth-Token": FOOTBALL_DATA_API_KEY}
    params = {"season": CURRENT_SEASON}  # no status filter — every fixture, played or not

    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()
    matches = response.json()["matches"]

    df = pd.DataFrame([{
        "match_id": m["id"],
        "utcDate": m["utcDate"],
        "matchday": m["matchday"],
        "home_team": m["homeTeam"]["name"],
        "away_team": m["awayTeam"]["name"],
    } for m in matches])

    df["utcDate"] = pd.to_datetime(df["utcDate"])
    return df.sort_values("utcDate").reset_index(drop=True)

full_season_fixtures = fetch_full_season_fixtures()
print(f"Total fixtures this season: {len(full_season_fixtures)}")
full_season_fixtures.head()

Total fixtures this season: 380


,match_id,utcDate,matchday,home_team,away_team
0,560542,2026-08-21 19:00:00+00:00,1,Arsenal FC,Coventry City FC
1,560543,2026-08-22 11:30:00+00:00,1,Hull City AFC,Manchester United FC
2,560544,2026-08-22 14:00:00+00:00,1,Ipswich Town FC,Sunderland AFC
3,560545,2026-08-22 14:00:00+00:00,1,Nottingham Forest FC,Leeds United FC
4,560546,2026-08-22 14:00:00+00:00,1,Everton FC,Crystal Palace FC


## Step 2: Build mid-season baseline state
Pulls this season's real finished matches so far and initializes each team's live state (form, goals, streaks) plus global head-to-head history.

In [3]:
# --- Recompute historical season standings from clean_matches (same tie-break logic as Step 4 in 04_feature_engineering.ipynb) ---
def build_all_team_matches(df):
    home_rows = df[["utcDate", "id", "season", "home_team_name", "home_goals", "away_goals", "winner"]].copy()
    home_rows.columns = ["utcDate", "match_id", "season", "team", "goals_scored", "goals_conceded", "winner"]
    home_rows["result"] = home_rows["winner"].map({"HOME_TEAM": "WIN", "AWAY_TEAM": "LOSS", "DRAW": "DRAW"})

    away_rows = df[["utcDate", "id", "season", "away_team_name", "away_goals", "home_goals", "winner"]].copy()
    away_rows.columns = ["utcDate", "match_id", "season", "team", "goals_scored", "goals_conceded", "winner"]
    away_rows["result"] = away_rows["winner"].map({"HOME_TEAM": "LOSS", "AWAY_TEAM": "WIN", "DRAW": "DRAW"})

    tm = pd.concat([home_rows, away_rows]).reset_index(drop=True)
    tm["points"] = tm["result"].map({"WIN": 3, "DRAW": 1, "LOSS": 0})
    tm["goal_difference"] = tm["goals_scored"] - tm["goals_conceded"]
    return tm

all_team_matches = build_all_team_matches(clean_matches)

season_points = (
    all_team_matches.groupby(["team", "season"])
    .agg(total_points=("points", "sum"), total_goal_diff=("goal_difference", "sum"))
    .reset_index()
)

season_order = (
    clean_matches.groupby("season")["utcDate"].min()
    .sort_values().reset_index().rename(columns={"utcDate": "season_start"})
)
season_order["season_rank"] = range(len(season_order))
season_points = season_points.merge(season_order, on="season")

season_points = season_points.sort_values(
    ["season", "total_points", "total_goal_diff"], ascending=[True, False, False]
)
season_points["final_position"] = season_points.groupby("season").cumcount() + 1

# Last season = the most recent one in your historical data (2025/26)
most_recent_rank = season_points["season_rank"].max()
last_season_positions = (
    season_points[season_points["season_rank"] == most_recent_rank]
    .set_index("team")["final_position"].to_dict()
)

# Promoted-team fallback — same derivation as 04_feature_engineering.ipynb
season_points["prev_season_rank"] = season_points["season_rank"] - 1
promoted_mask = ~season_points.apply(
    lambda row: ((season_points["team"] == row["team"]) &
                 (season_points["season_rank"] == row["prev_season_rank"])).any(),
    axis=1
)
promoted_mask &= season_points["season_rank"] > 0
promoted_avg_position = season_points.loc[promoted_mask, "final_position"].mean()

print(f"Last season positions known for {len(last_season_positions)} teams")
print(f"Promoted fallback: {promoted_avg_position:.2f}")


# --- Zero-init every team for this season (no real in-season data used) ---
def init_zero_team_state(team):
    return {
        "recent_points": [], "win_streak": 0, "loss_streak": 0,
        "goals_scored_sum": 0, "goals_scored_count": 0,
        "goals_conceded_sum": 0, "goals_conceded_count": 0,
        "clean_sheet_sum": 0, "clean_sheet_count": 0,
        "last_match_date": None,
        "home_win_sum": 0, "home_win_count": 0,
        "away_win_sum": 0, "away_win_count": 0,
        "total_points": 0,
        "prior_season_position": last_season_positions.get(team, promoted_avg_position),
    }

all_teams_full_season = pd.unique(pd.concat([full_season_fixtures["home_team"], full_season_fixtures["away_team"]]))
preseason_team_state = {team: init_zero_team_state(team) for team in all_teams_full_season}

print(f"Teams tracked: {len(preseason_team_state)}")
{team: round(preseason_team_state[team]["prior_season_position"], 1) for team in list(preseason_team_state)[:6]}

Last season positions known for 20 teams
Promoted fallback: 16.17
Teams tracked: 20


{'Arsenal FC': 1,
 'Hull City AFC': np.float64(16.2),
 'Ipswich Town FC': np.float64(16.2),
 'Nottingham Forest FC': 16,
 'Everton FC': 13,
 'Brentford FC': 9}

In [4]:
from collections import defaultdict

CURRENT_SEASON_INT = int(CURRENT_SEASON)

# --- Restrict to this season's real, already-played matches ---
def fetch_finished_matches_this_season():
    url = "https://api.football-data.org/v4/competitions/PL/matches"
    headers = {"X-Auth-Token": FOOTBALL_DATA_API_KEY}
    params = {"season": CURRENT_SEASON, "status": "FINISHED"}

    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()
    matches = response.json()["matches"]

    df = pd.DataFrame([{
        "id": m["id"],
        "utcDate": m["utcDate"],
        "home_team_name": m["homeTeam"]["name"],
        "away_team_name": m["awayTeam"]["name"],
        "home_goals": m["score"]["fullTime"]["home"],
        "away_goals": m["score"]["fullTime"]["away"],
        "winner": m["score"]["winner"],
    } for m in matches])

    df["utcDate"] = pd.to_datetime(df["utcDate"])
    return df.sort_values("utcDate").reset_index(drop=True)

season_matches = fetch_finished_matches_this_season()

def build_team_matches(df):
    """Same home/away melt as 04_feature_engineering.ipynb, scoped to one season's real matches."""
    home_rows = df[["utcDate", "id", "home_team_name", "home_goals", "away_goals", "winner"]].copy()
    home_rows.columns = ["utcDate", "match_id", "team", "goals_scored", "goals_conceded", "winner"]
    home_rows["is_home"] = True
    home_rows["result"] = home_rows["winner"].map({"HOME_TEAM": "WIN", "AWAY_TEAM": "LOSS", "DRAW": "DRAW"})

    away_rows = df[["utcDate", "id", "away_team_name", "away_goals", "home_goals", "winner"]].copy()
    away_rows.columns = ["utcDate", "match_id", "team", "goals_scored", "goals_conceded", "winner"]
    away_rows["is_home"] = False
    away_rows["result"] = away_rows["winner"].map({"HOME_TEAM": "LOSS", "AWAY_TEAM": "WIN", "DRAW": "DRAW"})

    tm = pd.concat([home_rows, away_rows]).sort_values(["team", "utcDate"]).reset_index(drop=True)
    tm["points"] = tm["result"].map({"WIN": 3, "DRAW": 1, "LOSS": 0})
    tm["clean_sheet"] = (tm["goals_conceded"] == 0).astype(int)
    tm["is_win"] = (tm["result"] == "WIN").astype(int)
    return tm

team_matches_season = build_team_matches(season_matches)

def init_team_state(team):
    rows = team_matches_season[team_matches_season["team"] == team].sort_values("utcDate")

    state = {
        "recent_points": list(rows["points"].tail(5)),          # for rolling_form_pts
        "win_streak": 0,
        "loss_streak": 0,
        "goals_scored_sum": rows["goals_scored"].sum(),
        "goals_scored_count": len(rows),
        "goals_conceded_sum": rows["goals_conceded"].sum(),
        "goals_conceded_count": len(rows),
        "clean_sheet_sum": rows["clean_sheet"].sum(),
        "clean_sheet_count": len(rows),
        "last_match_date": rows["utcDate"].max() if len(rows) else None,
        "home_win_sum": rows.loc[rows["is_home"], "is_win"].sum(),
        "home_win_count": rows["is_home"].sum(),
        "away_win_sum": rows.loc[~rows["is_home"], "is_win"].sum(),
        "away_win_count": (~rows["is_home"]).sum(),
        "total_points" : rows["points"].sum(),
        "prior_season_position": last_season_positions.get(team, promoted_avg_position),
    }

    # Recompute streaks by walking results in order (mirrors compute_streak from Phase 4)
    streak_w, streak_l = 0, 0
    for r in rows["result"]:
        streak_w = streak_w + 1 if r == "WIN" else 0
        streak_l = streak_l + 1 if r == "LOSS" else 0
    state["win_streak"], state["loss_streak"] = streak_w, streak_l

    return state

all_teams = pd.unique(team_matches_season["team"])
team_state = {team: init_team_state(team) for team in all_teams}

# --- Global head-to-head state (all historical seasons, per Phase 4's non-season-scoped h2h) ---
h2h_state = defaultdict(lambda: {"win_sum": 0, "count": 0})

for _, m in clean_matches.sort_values("utcDate").iterrows():
    home, away = m["home_team_name"], m["away_team_name"]
    home_win = 1 if m["home_goals"] > m["away_goals"] else 0
    away_win = 1 if m["away_goals"] > m["home_goals"] else 0
    h2h_state[(home, away)]["win_sum"] += home_win
    h2h_state[(home, away)]["count"] += 1
    h2h_state[(away, home)]["win_sum"] += away_win
    h2h_state[(away, home)]["count"] += 1

## Step 3: Poisson goal model
Generates realistic scorelines consistent with a predicted Win/Draw/Loss outcome, with shrinkage toward league averages for small samples.

In [5]:
import numpy as np

LEAGUE_AVG_GOALS = 1.4  # rough PL-wide average goals/team/match, used as fallback for early-season teams
SHRINKAGE_K = 5  # "games worth" of trust given to the league-average baseline
LEAGUE_AVG_PPG = 1.35              # rough PL-wide average points per team per match
LEAGUE_AVG_CLEAN_SHEET_PCT = 0.28  # rough PL-wide average clean sheet rate

def get_feature_value(state, key_sum, key_count, default=LEAGUE_AVG_GOALS):
    """Safe average with a sane fallback when a team has little/no history yet."""
    count = state[key_count]
    if count == 0:
        return default
    return state[key_sum] / count

def shrunk_average(total_sum, count, baseline, k=SHRINKAGE_K):
    return (total_sum + k * baseline) / (count + k)

def shrunk_rolling_form(recent_points, k=SHRINKAGE_K, league_avg_ppg=LEAGUE_AVG_PPG):
    """Shrink the implied points-per-game rate toward league average,
    then rescale back to a 'sum over up to 5 games' shape, matching
    how rolling_form_pts was originally defined in training."""
    count = len(recent_points)
    total = sum(recent_points)
    shrunk_ppg = shrunk_average(total, count, league_avg_ppg, k)
    return shrunk_ppg * count
    
def expected_goals(home_state, away_state):
    home_attack = shrunk_average(home_state["goals_scored_sum"], home_state["goals_scored_count"], LEAGUE_AVG_GOALS)
    away_defense = shrunk_average(away_state["goals_conceded_sum"], away_state["goals_conceded_count"], LEAGUE_AVG_GOALS)
    away_attack = shrunk_average(away_state["goals_scored_sum"], away_state["goals_scored_count"], LEAGUE_AVG_GOALS)
    home_defense = shrunk_average(home_state["goals_conceded_sum"], home_state["goals_conceded_count"], LEAGUE_AVG_GOALS)

    home_lambda = (home_attack + away_defense) / 2
    away_lambda = (away_attack + home_defense) / 2

    home_lambda *= 1.1  # home advantage bump

    return max(home_lambda, 0.1), max(away_lambda, 0.1)


def sample_scoreline(home_state, away_state, target_outcome, max_attempts=200):
    """
    target_outcome: 'WIN', 'DRAW', or 'LOSS' from the HOME team's perspective.
    Resamples from Poisson(home_lambda)/Poisson(away_lambda) until the
    scoreline agrees with the classifier's chosen outcome.
    """
    home_lambda, away_lambda = expected_goals(home_state, away_state)

    for _ in range(max_attempts):
        home_goals = np.random.poisson(home_lambda)
        away_goals = np.random.poisson(away_lambda)

        if target_outcome == "WIN" and home_goals > away_goals:
            return home_goals, away_goals
        if target_outcome == "LOSS" and home_goals < away_goals:
            return home_goals, away_goals
        if target_outcome == "DRAW" and home_goals == away_goals:
            return home_goals, away_goals

    # Fallback: force a minimal valid scoreline for this outcome (extremely rare to hit this)
    if target_outcome == "WIN":
        return 1, 0
    if target_outcome == "LOSS":
        return 0, 1
    return 0, 0

## Step 4: Feature builder and state updater
Builds the exact 18-feature row the model expects for a given matchup, and updates a team's state after a simulated result.

In [6]:
FEATURE_ORDER = [
    "home_rolling_form_pts", "home_win_streak", "home_loss_streak",
    "home_avg_goals_scored", "home_avg_goals_conceded", "home_clean_sheet_pct",
    "home_days_since_last_match", "home_h2h_win_pct", "home_prior_season_position",
    "away_rolling_form_pts", "away_win_streak", "away_loss_streak",
    "away_avg_goals_scored", "away_avg_goals_conceded", "away_clean_sheet_pct",
    "away_days_since_last_match", "away_h2h_win_pct", "away_prior_season_position",
]

OUTCOME_MAP = {0: "LOSS", 1: "DRAW", 2: "WIN"}  # from the home team's perspective

def get_h2h_win_pct(h2h, home, away):
    rec = h2h[(home, away)]
    if rec["count"] == 0:
        return 0.5
    return rec["win_sum"] / rec["count"]


def build_feature_row(state, h2h, home, away, match_date):
    hs, as_ = state[home], state[away]

    def days_since(s):
        if s["last_match_date"] is None:
            return 7
        return (match_date - s["last_match_date"]).days

    return {
        "home_rolling_form_pts": shrunk_rolling_form(hs["recent_points"]),
        "home_win_streak": hs["win_streak"],
        "home_loss_streak": hs["loss_streak"],
        "home_avg_goals_scored": shrunk_average(hs["goals_scored_sum"], hs["goals_scored_count"], LEAGUE_AVG_GOALS),
        "home_avg_goals_conceded": shrunk_average(hs["goals_conceded_sum"], hs["goals_conceded_count"], LEAGUE_AVG_GOALS),
        "home_clean_sheet_pct": shrunk_average(hs["clean_sheet_sum"], hs["clean_sheet_count"], LEAGUE_AVG_CLEAN_SHEET_PCT),
        "home_days_since_last_match": days_since(hs),
        "home_h2h_win_pct": get_h2h_win_pct(h2h, home, away),
        "home_prior_season_position": hs["prior_season_position"],

        "away_rolling_form_pts": shrunk_rolling_form(as_["recent_points"]),
        "away_win_streak": as_["win_streak"],
        "away_loss_streak": as_["loss_streak"],
        "away_avg_goals_scored": shrunk_average(as_["goals_scored_sum"], as_["goals_scored_count"], LEAGUE_AVG_GOALS),
        "away_avg_goals_conceded": shrunk_average(as_["goals_conceded_sum"], as_["goals_conceded_count"], LEAGUE_AVG_GOALS),
        "away_clean_sheet_pct": shrunk_average(as_["clean_sheet_sum"], as_["clean_sheet_count"], LEAGUE_AVG_CLEAN_SHEET_PCT),
        "away_days_since_last_match": days_since(as_),
        "away_h2h_win_pct": get_h2h_win_pct(h2h, away, home),
        "away_prior_season_position": as_["prior_season_position"],
    }


def update_team_state(state, points, result, goals_scored, goals_conceded, is_home, match_date):
    state["recent_points"] = (state["recent_points"] + [points])[-5:]
    state["total_points"] = state.get("total_points", 0) + points
    state["win_streak"] = state["win_streak"] + 1 if result == "WIN" else 0
    state["loss_streak"] = state["loss_streak"] + 1 if result == "LOSS" else 0
    state["goals_scored_sum"] += goals_scored
    state["goals_scored_count"] += 1
    state["goals_conceded_sum"] += goals_conceded
    state["goals_conceded_count"] += 1
    state["clean_sheet_sum"] += 1 if goals_conceded == 0 else 0
    state["clean_sheet_count"] += 1
    state["last_match_date"] = match_date
    if is_home:
        state["home_win_sum"] += 1 if result == "WIN" else 0
        state["home_win_count"] += 1
    else:
        state["away_win_sum"] += 1 if result == "WIN" else 0
        state["away_win_count"] += 1


## Step 5: Monte Carlo engine + mid-season simulation
Defines `run_monte_carlo` (batched across all simulations for speed) and `compute_final_table`, then runs 1,000 simulated seasons starting from today's real standings.

In [7]:
import time

N_SIMULATIONS = 1000

def compute_final_table(state):
    """Build a ranked final table from one simulated season's end state."""
    table = []
    for team, s in state.items():
        table.append({
            "team": team,
            "points": s["total_points"],
            "goal_diff": s["goals_scored_sum"] - s["goals_conceded_sum"],
            "goals_scored": s["goals_scored_sum"],
        })
    df = pd.DataFrame(table)
    # Standard PL tie-break order: points, then goal difference, then goals scored
    df = df.sort_values(["points", "goal_diff", "goals_scored"], ascending=False).reset_index(drop=True)
    df["position"] = df.index + 1
    return df


def run_monte_carlo(fixtures, initial_team_state, initial_h2h_state, n_sims=1000):
    states = []
    for _ in range(n_sims):
        s = {t: dict(v) for t, v in initial_team_state.items()}
        for v in s.values():
            v["recent_points"] = list(v["recent_points"])
        states.append(s)

    h2hs = [defaultdict(lambda: {"win_sum": 0, "count": 0},
                         {k: dict(v) for k, v in initial_h2h_state.items()})
            for _ in range(n_sims)]

    for _, fx in fixtures.iterrows():
        home, away, match_date = fx["home_team"], fx["away_team"], fx["utcDate"]

        rows = [build_feature_row(states[i], h2hs[i], home, away, match_date) for i in range(n_sims)]
        X = pd.DataFrame(rows)[FEATURE_ORDER]

        proba = model.predict_proba(X)

        draws = np.random.rand(n_sims, 1)
        cumprobs = np.cumsum(proba, axis=1)
        outcome_codes = (draws < cumprobs).argmax(axis=1)

        for i in range(n_sims):
            outcome = OUTCOME_MAP[outcome_codes[i]]
            home_goals, away_goals = sample_scoreline(states[i][home], states[i][away], outcome)

            home_points = {"WIN": 3, "DRAW": 1, "LOSS": 0}[outcome]
            away_result = {"WIN": "LOSS", "DRAW": "DRAW", "LOSS": "WIN"}[outcome]
            away_points = {"WIN": 3, "DRAW": 1, "LOSS": 0}[away_result]

            update_team_state(states[i][home], home_points, outcome, home_goals, away_goals, True, match_date)
            update_team_state(states[i][away], away_points, away_result, away_goals, home_goals, False, match_date)

            winner = home if outcome == "WIN" else away if outcome == "LOSS" else None
            loser = away if outcome == "WIN" else home if outcome == "LOSS" else None
            if winner:
                h2hs[i][(winner, loser)]["win_sum"] += 1
            h2hs[i][(home, away)]["count"] += 1
            h2hs[i][(away, home)]["count"] += 1

    return states


start = time.time()
final_states = run_monte_carlo(remaining_fixtures, team_state, h2h_state, n_sims=N_SIMULATIONS)
elapsed = time.time() - start
print(f"Ran {N_SIMULATIONS} simulations in {elapsed:.1f}s")

all_final_tables = []
for sim, state in enumerate(final_states):
    ft = compute_final_table(state)
    ft["sim"] = sim
    all_final_tables.append(ft)

all_sims_df = pd.concat(all_final_tables, ignore_index=True)
all_sims_df.head()

Ran 1000 simulations in 53.2s


,team,points,goal_diff,goals_scored,position,sim
0,Arsenal FC,83,33,64,1,0
1,Aston Villa FC,77,32,74,2,0
2,Sunderland AFC,77,30,74,3,0
3,Liverpool FC,68,21,70,4,0
4,Manchester City FC,67,-1,69,5,0


## Step 6: Summarize mid-season outlook
Aggregates the 1,000 simulated final tables into per-team title/top-4/relegation probabilities.

In [8]:
def summarize_simulations(all_sims_df, n_sims):
    summary = (
        all_sims_df.groupby("team")
        .agg(
            avg_points=("points", "mean"),
            avg_position=("position", "mean"),
            title_pct=("position", lambda x: (x == 1).mean() * 100),
            top4_pct=("position", lambda x: (x <= 4).mean() * 100),
            relegation_pct=("position", lambda x: (x >= 18).mean() * 100),
        )
        .sort_values("avg_position")
        .reset_index()
    )
    return summary

season_outlook = summarize_simulations(all_sims_df, N_SIMULATIONS)
season_outlook

,team,avg_points,avg_position,title_pct,top4_pct,relegation_pct
0,Arsenal FC,72.453,2.894,38.9,81.0,0.0
1,Manchester City FC,70.511,3.325,28.6,75.5,0.0
2,Liverpool FC,60.778,6.568,7.7,39.9,1.5
3,Manchester United FC,60.415,6.834,5.5,35.2,1.3
4,Brighton & Hove Albion FC,56.513,8.390,3.3,24.0,3.5
5,Brentford FC,55.512,8.929,3.1,20.4,3.9
6,Sunderland AFC,55.098,9.125,2.1,19.0,5.1
7,AFC Bournemouth,53.974,9.627,2.0,17.6,6.7
8,Aston Villa FC,53.878,9.767,1.3,14.2,5.5
9,Chelsea FC,53.700,9.811,1.5,15.4,5.5


## Step 7: Preseason simulation
Same engine, but starting every team from a blank slate (prior-season position only) instead of real in-season form — isolates how much of the mid-season outlook is driven by early-season results.

In [9]:
start = time.time()
preseason_final_states = run_monte_carlo(full_season_fixtures, preseason_team_state, h2h_state, n_sims=N_SIMULATIONS)
elapsed = time.time() - start
print(f"Ran {N_SIMULATIONS} preseason simulations in {elapsed:.1f}s")

preseason_all_tables = []
for sim, state in enumerate(preseason_final_states):
    ft = compute_final_table(state)
    ft["sim"] = sim
    preseason_all_tables.append(ft)

preseason_all_sims_df = pd.concat(preseason_all_tables, ignore_index=True)

preseason_outlook = summarize_simulations(preseason_all_sims_df, N_SIMULATIONS)
preseason_outlook

Ran 1000 preseason simulations in 30.6s


,team,avg_points,avg_position,title_pct,top4_pct,relegation_pct
0,Arsenal FC,65.732,5.023,20.9,56.6,0.4
1,Manchester City FC,64.932,5.204,18.5,54.7,0.8
2,Manchester United FC,61.475,6.537,11.6,40.4,1.8
3,Aston Villa FC,60.476,6.969,8.9,39.0,2.0
4,Liverpool FC,60.174,7.180,8.2,37.1,2.6
5,AFC Bournemouth,57.718,8.128,5.8,27.4,3.9
6,Sunderland AFC,56.016,8.913,5.4,23.8,6.1
7,Brighton & Hove Albion FC,55.398,9.173,4.8,22.6,6.1
8,Brentford FC,54.010,9.783,3.4,18.1,8.6
9,Chelsea FC,52.886,10.276,3.2,17.6,11.5


## Step 8: Mid-season vs. preseason comparison

In [10]:
comparison = season_outlook[["team", "avg_position", "top4_pct"]].merge(
    preseason_outlook[["team", "avg_position", "top4_pct"]],
    on="team", suffixes=("_midseason", "_preseason")
)
comparison.sort_values("avg_position_midseason")

,team,avg_position_midseason,top4_pct_midseason,avg_position_preseason,top4_pct_preseason
0,Arsenal FC,2.894,81.0,5.023,56.6
1,Manchester City FC,3.325,75.5,5.204,54.7
2,Liverpool FC,6.568,39.9,7.180,37.1
3,Manchester United FC,6.834,35.2,6.537,40.4
4,Brighton & Hove Albion FC,8.390,24.0,9.173,22.6
5,Brentford FC,8.929,20.4,9.783,18.1
6,Sunderland AFC,9.125,19.0,8.913,23.8
7,AFC Bournemouth,9.627,17.6,8.128,27.4
8,Aston Villa FC,9.767,14.2,6.969,39.0
9,Chelsea FC,9.811,15.4,10.276,17.6


## Step 9: Save results

In [11]:
import os
os.makedirs("../data/predictions", exist_ok=True)

season_outlook.to_csv("../data/predictions/season_simulation_summary.csv", index=False)
all_sims_df.to_csv("../data/predictions/season_simulation_raw.csv", index=False)

preseason_outlook.to_csv("../data/predictions/preseason_simulation_summary.csv", index=False)
preseason_all_sims_df.to_csv("../data/predictions/preseason_simulation_raw.csv", index=False)

comparison.to_csv("../data/predictions/midseason_vs_preseason_comparison.csv", index=False)

print("Saved mid-season, preseason, and comparison simulation results.")

Saved mid-season, preseason, and comparison simulation results.
